# 09 - How fragile are the allocation decisions?

Step 9.1 of `THESIS_GUIDE.md` (PIANO.md C4). Notebook 04 showed that two seeds of the same model share only part of their portfolio. This notebook asks why, by comparing two quantities on the same stock and day (`xaifin/portfolio/fragility.py`):

- the **margin** of a held stock to the top-k boundary, in standard deviations of the day's scores (`margin_z`, chapter 7): how far its score could fall before another stock overtakes it;
- the **seed noise** of the stock-day: the standard deviation, across the seeds of the same model, of the stock's daily z-scored score, i.e. how much training luck moves its position in the cross-section.

A held stock whose margin is smaller than its seed noise (ratio < 1) is held or not depending on the seed: the decision is fragile. If the measure is right, the share of such holdings should match the share of the portfolio that actually changes between two seeds.

Core grid, T = 20, L = 5, every seed available.

In [1]:
import warnings
from itertools import combinations

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

from xaifin import viz
from xaifin.config import CORE_UNIVERSES, RESULTS_ROOT, TOP_K, UNIVERSE_NAMES
from xaifin.portfolio.backtest import find_portfolios
from xaifin.portfolio.fragility import fragility, seed_noise, summary, zscores
from xaifin.portfolio.topk import overlap

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda v: f"{v:.3f}")
SEQ_LEN, PRED_LEN = 20, 5
portfolios = find_portfolios()
portfolios = portfolios[(portfolios["seq_len"] == SEQ_LEN) & (portfolios["pred_len"] == PRED_LEN)
                        & portfolios["universe"].isin(CORE_UNIVERSES) & (portfolios["k"] == portfolios["universe"].map(TOP_K))]
portfolios = portfolios[portfolios["path"].map(lambda p: (p / "holdings.parquet").exists())]
print(len(portfolios), "portfolios with holdings")

54 portfolios with holdings


## 1. Margins against seed noise

Per model and universe: the median seed noise and margin of the held stocks, the median ratio, the share of holdings closer to the boundary than their seed noise (and than half of it), and, for comparison, the turnover actually observed between the portfolios of two seeds (mean over pairs of seeds and rebalances).

In [2]:
rows, held_all = [], []
for (model, universe), group in portfolios.groupby(["model", "universe"]):
    seeds = sorted(group["seed"])
    if len(seeds) < 2:
        continue
    noise = seed_noise(zscores(model, universe, SEQ_LEN, PRED_LEN, seeds=seeds))
    holdings = {r.seed: pd.read_parquet(r.path / "holdings.parquet") for r in group.itertuples()}
    held = pd.concat([fragility(h, noise).assign(seed=s) for s, h in holdings.items()])
    held_all.append(held.assign(model=model, universe=universe))
    observed = np.mean([overlap(holdings[a], holdings[b])["turnover"].mean() for a, b in combinations(seeds, 2)])
    rows.append({"model": model, "universe": universe, "seeds": len(seeds), **summary(held), "observed seed turnover": observed})
table = pd.DataFrame(rows)
table["model"] = pd.Categorical(table["model"], viz.MODEL_ORDER)
table = table.sort_values(["universe", "model"]).reset_index(drop=True)
table

,model,universe,seeds,holdings,median_noise,median_margin,median_ratio,share_below_noise,share_below_half_noise,observed seed turnover
0,MASTER,dji,3,3900,0.598,0.438,0.755,0.611,0.357,0.571
1,MATCC,dji,3,3900,0.726,0.469,0.664,0.662,0.388,0.635
2,FactorVAE,dji,3,3885,0.369,0.408,1.114,0.466,0.267,0.401
3,HIST,dji,3,3900,0.378,0.437,1.049,0.483,0.287,0.432
4,DiscoverPLF,dji,3,3900,0.504,0.464,0.956,0.520,0.297,0.469
5,FinFormer,dji,3,3900,0.683,0.307,0.481,0.756,0.514,0.689
6,MASTER,nasdaq100,3,7800,0.656,0.404,0.634,0.672,0.416,0.618
7,MATCC,nasdaq100,3,7800,0.856,0.416,0.515,0.768,0.489,0.721
8,FactorVAE,nasdaq100,3,7770,0.440,0.376,0.868,0.548,0.320,0.469
9,HIST,nasdaq100,3,7800,0.627,0.509,0.847,0.557,0.335,0.487


## 2. Does the measure predict what happens?

Each point is one model in one universe: the share of its holdings with a margin below their seed noise against the share of the portfolio that changes from one seed to another. Points on the diagonal mean that the margins and the seed noise explain the instability of the portfolios between seeds.

In [3]:
fig = px.scatter(table, x="share_below_noise", y="observed seed turnover", color="model", symbol="universe",
                 color_discrete_sequence=viz.THEMES[False]["series"], category_orders={"model": viz.MODEL_ORDER},
                 labels={"share_below_noise": "holdings with margin < seed noise", "observed seed turnover": "turnover between seeds"},
                 title="Fragile holdings against the portfolio change between seeds")
fig.add_shape(type="line", x0=0, y0=0, x1=1, y1=1, line=dict(color=viz.THEMES[False]["muted"], dash="dot", width=1))
fig.update_xaxes(range=[0, 1], tickformat=".0%")
fig.update_yaxes(range=[0, 1], tickformat=".0%")
fig.show()
print("correlation:", round(table["share_below_noise"].corr(table["observed seed turnover"]), 3),
      "| mean absolute gap:", round((table["share_below_noise"] - table["observed seed turnover"]).abs().mean(), 3))

correlation: 0.962 | mean absolute gap: 0.063


## 3. Distribution of the ratio margin / seed noise

All held stock-days of a model, pooled over universes and seeds; the dotted line at 1 separates the decisions that a new seed can overturn.

In [4]:
held_all = pd.concat(held_all)
fig = px.histogram(held_all[held_all["ratio"] < 5], x="ratio", color="model", nbins=50, histnorm="probability",
                   facet_col="model", facet_col_wrap=3, category_orders={"model": viz.MODEL_ORDER},
                   color_discrete_sequence=viz.THEMES[False]["series"], title="Margin / seed noise of the held stocks (cut at 5)")
fig.add_vline(x=1, line_dash="dot", line_color=viz.THEMES[False]["muted"])
fig.update_layout(showlegend=False)
fig.show()
held_all.groupby("model")["ratio"].describe(percentiles=[0.1, 0.25, 0.5, 0.75]).reindex(viz.MODEL_ORDER)

,count,mean,std,min,10%,25%,50%,75%,max
model,,,,,,,,,
MASTER,17940.000,1.249,2.325,0.000,0.118,0.320,0.730,1.429,109.342
MATCC,17940.000,0.971,1.964,0.000,0.102,0.269,0.611,1.156,116.131
FactorVAE,17865.000,1.815,4.035,0.000,0.166,0.436,1.004,2.048,156.653
HIST,17940.000,1.749,3.884,0.000,0.137,0.367,0.855,1.839,132.863
DiscoverPLF,17940.000,1.532,3.013,0.000,0.145,0.384,0.879,1.732,103.125
FinFormer,17940.000,1.114,2.301,0.000,0.093,0.252,0.603,1.262,70.236


## 4. Saved summary

`results/analysis/decision_fragility.csv`: the table of section 1.

In [5]:
table.to_csv(RESULTS_ROOT / "analysis" / "decision_fragility.csv", index=False)
print("saved")

saved


## Observations

18 model-universe pairs of the core grid at L = 5, three seeds each.

1. **About half or more of the held positions are within training luck of the boundary.** The share of holdings whose margin to the top-k boundary is smaller than their seed noise ranges from 46% (FactorVAE on EURO STOXX 50) to 77% (MATCC on NASDAQ-100); 25-51% are even within half of it. The median ratio margin / seed noise is below 1 for MASTER, MATCC and FinFormer in every universe, and around 1 for the others.
2. **The measure predicts what happens between seeds.** Across the 18 pairs, the share of fragile holdings and the share of the portfolio that actually changes between two seeds are correlated at 0.96. With three seeds the fragile share is about 6 points higher than the observed turnover (46-77% against 40-72%), because the spread of three seeds is larger than the difference between two of them; with two seeds (first run of this notebook, seeds 0 and 5) the two quantities matched within 1.4 points.
3. **The ranking follows the models' seed behaviour.** FactorVAE's portfolios are the least fragile (46-55% of fragile holdings, 40-47% turnover between seeds), because its selection rule keeps an almost untrained model whose seeds agree; MATCC (63-77%) and FinFormer (63-76%) are the most fragile.

The instability of the portfolios between seeds is therefore not a separate phenomenon: it is the seed noise of the scores acting on positions that are close to the boundary. A model's choice of stocks is reliable only for the few positions whose margin is well above the noise of its training.